# Video prep v2 — Colab ⇄ Claude Code (retakes by Claude, optional Omni Flash + background swap)

Works for **vertical (9:16)** and **horizontal (16:9)** videos. Runtime → Change runtime type → **T4 GPU**.
Drive: put the clip in `My Drive/video-editor/raw footage/`; share `My Drive/video-editor/results/` once as
*Anyone with the link*.

| # | Who | What |
|---|---|---|
| 1 | You | upload the phone clip (+ the mic .m4a if you used one), set **Step 0**, run **Part A** (silences + breaths removed, transcription, list of possible retakes — nothing cut yet) |
| 2 | **Claude Code** | send it the `results/<slug>` link: *"retakes for <slug>"* → Claude picks the best takes and writes `final_cut.json` into that folder |
| 3 | You | run **Part B** (final cut; with `OMNI_FLASH`: ≤15 s clips in `omni/to_omni/`) |
| 4 | You (only with Omni) | edit each clip in Omni Flash, save them in `omni/from_omni/` with the SAME names (`clip_01.mp4` …) |
| 5 | You | run **Part C** (joins Omni clips + your original voice → background swap if ticked → soft grade → `edit.mp4`) |
| 6 | **Claude Code** | *"finish <slug>"* → captions, Purple Room graphics, sounds, face check, render |

No Omni and no background? Skip Parts B and C: after step 2 Claude does everything.
Models: Darija `anaszil/whisper-large-v3-turbo-darija` · EN/FR `openai/whisper-large-v3-turbo` · cut-out MatAnyone 2
(non-commercial licence) + BiRefNet-portrait + LaMa.

In [ ]:
#@title Step 0 · Settings (run this first, every time you open the notebook)
SLUG = "my-video"  #@param {type:"string"}
#@markdown **Exact file names** in `video-editor/raw footage/` (copy them from Drive, extension included).
#@markdown Video (e.g. `IMG_1234.MOV`). Empty = the newest video there.
VIDEO_NAME = ""  #@param {type:"string"}
ROOT = "/content/drive/MyDrive/video-editor"  #@param {type:"string"}
#@markdown Mic recording to use INSTEAD of the phone audio (e.g. `Recording (12).m4a`). Synced automatically
#@markdown (offset + clock drift). Empty = keep the phone audio · `auto` = find the matching recording by itself.
MIC_AUDIO = ""  #@param {type:"string"}

#@markdown ---
#@markdown **Options**
#@markdown Change your clothes etc. in Omni Flash (exports ≤15 s clips after the final cut).
OMNI_FLASH = False  #@param {type:"boolean"}
OMNI_MAX_SECONDS = 15  #@param {type:"number"}
#@markdown Replace the background (MatAnyone 2), after Omni if both are ticked.
CHANGE_BACKGROUND = False  #@param {type:"boolean"}
#@markdown Your AI background image (empty = `results/<slug>/background.png`). Tick if you're still in that image.
BACKGROUND_IMAGE = ""  #@param {type:"string"}
BACKGROUND_HAS_ME = True  #@param {type:"boolean"}
MATTING_SIZE = 1080  #@param [720, 1080] {type:"raw"}
COLOR_MATCH = 0.25  #@param {type:"slider", min:0, max:1, step:0.05}
LIGHT_WRAP = 0.35  #@param {type:"slider", min:0, max:1, step:0.05}
BACKGROUND_BLUR = 1.5  #@param {type:"slider", min:0, max:8, step:0.5}
GRAIN = 1.0  #@param {type:"slider", min:0, max:2, step:0.1}
EDGE_TIGHTEN = 0.0  #@param {type:"slider", min:0, max:0.3, step:0.02}

import os
RES = f"{ROOT}/results/{SLUG}"
if not BACKGROUND_IMAGE:
    BACKGROUND_IMAGE = f"{RES}/background.png"
OUTPUT_SIZE = "1080p"
print("Settings saved. Results folder:", RES)

In [ ]:
#@title Part A · Prep: convert, silences + breaths, transcription (Darija + EN/FR), retake CANDIDATES (≈12–18 min / 5 min of video)
import subprocess, time, json, os
t0 = time.time()
if not subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip():
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save, then run Step 0 and this cell again.")
from google.colab import drive
drive.mount("/content/drive")
!wget -q -O /content/colab_prep.py https://raw.githubusercontent.com/anassajrx/video-editorV/claude/hopeful-clarke-qhnr9z/local/colab/colab_prep.py
!wget -q -O /content/mic_sync.py https://raw.githubusercontent.com/anassajrx/video-editorV/claude/hopeful-clarke-qhnr9z/local/colab/mic_sync.py
flags = f'--slug "{SLUG}" --root "{ROOT}" --no-retakes --no-grade'
if VIDEO_NAME: flags += f' --video "{VIDEO_NAME}"'
if MIC_AUDIO.strip(): flags += f' --mic "{MIC_AUDIO.strip()}"'
!cd /content && python colab_prep.py {flags}
assert os.path.exists(f"{RES}/phrases.json"), "Part A did not finish: read the error above (or send it to Claude)."
meta = json.load(open(f"{RES}/meta.json")); meta.update({"omniFlash": OMNI_FLASH, "backgroundSwapRequested": CHANGE_BACKGROUND, "stage": "waiting-for-claude-retakes"})
json.dump(meta, open(f"{RES}/meta.json", "w"), indent=1)
print(f"\nPart A done in {(time.time() - t0) / 60:.1f} min · {meta['width']}x{meta['height']} {meta['orientation']}")
print(f'''
NEXT → Claude Code: send the link of the folder results/{SLUG} with: "retakes for {SLUG}"
       Claude writes final_cut.json into that folder.''')
if OMNI_FLASH or CHANGE_BACKGROUND:
    print("       Then come back here and run Part B.")
else:
    print("       No Omni / no background: Claude does the rest, you're done with Colab.")

## Part B · after Claude wrote `final_cut.json`

In [ ]:
#@title Part B · Final cut from Claude's final_cut.json (+ Omni Flash clips) (≈3–5 min)
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(f"{RES}/final_cut.json"), "final_cut.json is not in the folder yet: ask Claude Code for the retakes first."
!wget -q -O /content/colab_cut.py https://raw.githubusercontent.com/anassajrx/video-editorV/claude/hopeful-clarke-qhnr9z/local/colab/colab_cut.py
flags = f'--slug "{SLUG}" --root "{ROOT}"' + (f" --omni --omni-max {OMNI_MAX_SECONDS}" if OMNI_FLASH else "")
!cd /content && python colab_cut.py {flags}
if OMNI_FLASH:
    print(f'''
NEXT → Omni Flash: edit each clip in {RES}/omni/to_omni/ (clip_01.mp4, clip_02.mp4, …).
       Save every result in {RES}/omni/from_omni/ with the SAME name (clip_01.mp4 …).
       Any clip you don't edit keeps the original. Then run Part C.''')
else:
    print("NEXT → run Part C" + (" (background swap + grade)." if CHANGE_BACKGROUND else " (grade)."))

## Part C · after Omni Flash (or right after Part B if you don't use Omni)

In [ ]:
#@title Part C1 · Join the Omni Flash clips (only if OMNI_FLASH) + choose the video for the next steps
import os
from google.colab import drive
drive.mount("/content/drive")
!wget -q -O /content/colab_finish.py https://raw.githubusercontent.com/anassajrx/video-editorV/claude/hopeful-clarke-qhnr9z/local/colab/colab_finish.py
if OMNI_FLASH:
    !cd /content && python colab_finish.py join --slug "{SLUG}" --root "{ROOT}"
    SWAP_INPUT = f"{RES}/edit_omni.mp4"
else:
    SWAP_INPUT = f"{RES}/edit_cut.mp4"
assert os.path.exists(SWAP_INPUT), f"Missing {SWAP_INPUT} (run Part B first)."
print("Video for the next step:", SWAP_INPUT)
if CHANGE_BACKGROUND:
    import subprocess, json
    meta = json.load(open(f"{RES}/meta.json"))
    frame = f"{RES}/frame_for_image_ai.png"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-ss", "1", "-i", SWAP_INPUT, "-frames:v", "1", frame], check=True)
    print(f'''Background swap: give {frame} to your image AI ("keep the person exactly the same, replace only the background
with [room], same framing, same size {meta['width']}x{meta['height']}, photorealistic"), save it as {BACKGROUND_IMAGE},
then run C2 → C2e. (Without the swap, jump to C3.)''')

## Part C2 · Background swap (only if `CHANGE_BACKGROUND`) — run C2 → C2e top to bottom
The cut-out (C2d) is the slow step (~20–40 min for ~3.5 min of video at 1080, about half at 720).

In [ ]:
#@title C2 · Setup for the background swap (about 1–2 minutes)
import os, sys, json, time, math, subprocess, shutil
from pathlib import Path
T_SETUP = time.time()

def run(cmd, quiet=True):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("Command failed: " + " ".join(map(str, cmd[:8])) + " ...")
    return r.stdout

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU found. Go to Runtime → Change runtime type → T4 GPU, click Save, then run this cell again.")
print("GPU:", gpu)

from google.colab import drive
drive.mount("/content/drive")
assert CHANGE_BACKGROUND, "CHANGE_BACKGROUND is off in Step 0: Part B is not needed."

MA2_DIR = Path("/content/MatAnyone2")
MA2_COMMIT = "0079197acd6d16a741f71558809c06c586c579e0"   # the version this notebook was tested with
if not MA2_DIR.exists():
    run(["git", "clone", "-q", "https://github.com/pq-yang/MatAnyone2", str(MA2_DIR)])
    run(["git", "-C", str(MA2_DIR), "checkout", "-q", MA2_COMMIT])
sys.path.insert(0, str(MA2_DIR))

# MatAnyone 2's own package install is broken with new build tools, so we use the code directly
# and only add what Colab doesn't already have. --no-deps keeps Colab's own versions untouched.
!pip -q install hydra-core omegaconf onnxruntime pooch pymatting
!pip -q install --no-deps rembg simple-lama-inpainting

import numpy as np, torch, torch.nn.functional as F
from PIL import Image
from IPython.display import display, HTML
DEV = torch.device("cuda")
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
print(f"Setup done in {time.time() - T_SETUP:.0f} s.")

In [ ]:
#@title C2b · Helpers (just run this cell)
def probe(path):
    out = run(["ffprobe", "-v", "error", "-show_entries",
               "stream=codec_type,width,height,avg_frame_rate,r_frame_rate,color_transfer,pix_fmt:"
               "stream_side_data=rotation:format=duration", "-of", "json", str(path)])
    j = json.loads(out)
    v = next(s for s in j["streams"] if s["codec_type"] == "video")
    rate = v.get("avg_frame_rate") or v.get("r_frame_rate") or "30/1"
    num, den = (rate.split("/") + ["1"])[:2]
    fps = float(num) / float(den) if float(den) else 30.0
    return dict(w=int(v["width"]), h=int(v["height"]), fps=fps,
                dur=float(j["format"].get("duration", 0)),
                audio=any(s["codec_type"] == "audio" for s in j["streams"]),
                transfer=v.get("color_transfer", ""), pix_fmt=v.get("pix_fmt", ""))

def to_seconds(t):
    s = 0.0
    for p in str(t).strip().split(":"):
        s = s * 60 + float(p)
    return s

READ_VF = "scale=in_color_matrix=bt709:in_range=tv"            # how we read our own clip
WRITE_VF = "scale=out_color_matrix=bt709:out_range=tv,format=yuv420p"
TAGS = ["-colorspace", "bt709", "-color_primaries", "bt709", "-color_trc", "bt709"]
X264 = ["-c:v", "libx264", "-preset", "veryfast", "-crf", "16", "-pix_fmt", "yuv420p"]

def read_frames(path, w, h, gray=False):
    """Yield frames one at a time (never the whole video in memory)."""
    ch = 1 if gray else 3
    cmd = ["ffmpeg", "-v", "error", "-i", str(path)]
    if not gray:
        cmd += ["-vf", READ_VF]
    cmd += ["-f", "rawvideo", "-pix_fmt", "gray" if gray else "rgb24", "-"]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=w * h * ch * 4)
    n = w * h * ch
    try:
        while True:
            b = p.stdout.read(n)
            if len(b) < n:
                break
            yield np.frombuffer(b, np.uint8).reshape(h, w, ch)
    finally:
        p.stdout.close(); p.wait()

def writer(path, w, h, fps, gray=False, audio_from=None):
    cmd = ["ffmpeg", "-v", "error", "-y", "-f", "rawvideo", "-pix_fmt", "gray" if gray else "rgb24",
           "-s", f"{w}x{h}", "-r", f"{fps:.6f}", "-i", "-"]
    if gray:
        cmd += ["-c:v", "ffv1", str(path)]          # lossless cut-out mask
    else:
        if audio_from:
            cmd += ["-i", str(audio_from), "-map", "0:v", "-map", "1:a?", "-c:a", "aac", "-b:a", "192k", "-shortest"]
        cmd += ["-vf", WRITE_VF, *X264, *TAGS, "-movflags", "+faststart", str(path)]
    return subprocess.Popen(cmd, stdin=subprocess.PIPE)

def gblur(x, sigma):
    """Fast separable Gaussian blur on a (C,H,W) GPU tensor."""
    if sigma <= 0.05:
        return x
    r = max(1, int(math.ceil(sigma * 3)))
    k = torch.arange(-r, r + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(-(k * k) / (2 * sigma * sigma)); k = k / k.sum()
    c = x.shape[0]
    y = F.pad(x[None], (r, r, 0, 0), mode="replicate")
    y = F.conv2d(y, k.view(1, 1, 1, -1).repeat(c, 1, 1, 1), groups=c)
    y = F.pad(y, (0, 0, r, r), mode="replicate")
    y = F.conv2d(y, k.view(1, 1, -1, 1).repeat(c, 1, 1, 1), groups=c)
    return y[0]

def big_blur(x, sigma):
    """Large blurs done at a quarter size (much faster, looks the same)."""
    if sigma < 6:
        return gblur(x, sigma)
    h, w = x.shape[-2:]
    s = F.interpolate(x[None], scale_factor=0.25, mode="area")[0]
    s = gblur(s, sigma / 4)
    return F.interpolate(s[None], size=(h, w), mode="bilinear", align_corners=False)[0]

_SEG = {}   # kept for compatibility (the person-finder now runs in its own process)
_PM_SCRIPT = """
import sys, numpy as np
from PIL import Image
from rembg import new_session, remove
s = new_session(sys.argv[3])
m = remove(Image.open(sys.argv[1]).convert("RGB"), session=s, only_mask=True)
np.save(sys.argv[2], np.array(m.convert("L")))
"""
def person_mask(img_rgb):
    """Soft mask of the person in one image (0..255), BiRefNet-portrait through rembg.
    Runs in a separate process: BiRefNet peaks at ~9.5 GB RAM on CPU, and the memory is only fully
    released when that process exits (Colab has ~12.7 GB)."""
    pth = WORK / "_pm_in.png"; outp = WORK / "_pm_out.npy"; scr = WORK / "_pm.py"
    scr.write_text(_PM_SCRIPT)
    Image.fromarray(img_rgb).save(pth)
    print("Finding you in the image (BiRefNet-portrait; first time downloads ~1 GB)...")
    r = subprocess.run([sys.executable, str(scr), str(pth), str(outp), SEG_MODEL], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("The person-finder failed (see the error above).")
    m = np.load(outp)
    return np.array(Image.fromarray(m).resize((img_rgb.shape[1], img_rgb.shape[0]), Image.BILINEAR))

SEG_MODEL = "birefnet-portrait"

def show_images(paths_or_arrays, width=220):
    from io import BytesIO
    import base64
    tags = []
    for item in paths_or_arrays:
        im = Image.open(item) if isinstance(item, (str, Path)) else Image.fromarray(item)
        im = im.convert("RGB"); im.thumbnail((width * 2, width * 4))
        buf = BytesIO(); im.save(buf, format="JPEG", quality=85)
        tags.append(f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}" '
                    f'style="width:{width}px;margin-right:8px;border-radius:6px">')
    display(HTML("".join(tags)))

print("Helpers loaded.")

In [ ]:
#@title C2c · Load the video (Omni result or final cut)
t0 = time.time()
src = Path(SWAP_INPUT)
assert src.exists(), f"Can't find {src}. Run C1 first."
OUT = Path(RES) / "background_files"; OUT.mkdir(exist_ok=True)
info = probe(src)
fps = info["fps"] if 1 < info["fps"] <= 120 else 30.0
fps = round(fps * 1000) / 1000
segment = WORK / "segment.mp4"
# whole cut video, BT.709-tagged (same frames, same timing as edit.mp4), long side max 1920 for both orientations
vf = ["scale='if(gt(iw,ih),min(1920,iw),-2)':'if(gt(iw,ih),-2,min(1920,ih))':flags=lanczos",
      "scale=out_color_matrix=bt709:out_range=tv,format=yuv420p"]
run(["ffmpeg", "-v", "error", "-y", "-i", str(src), "-vf", ",".join(vf), "-vsync", "cfr", "-r", f"{fps}", *X264, *TAGS,
     "-c:a", "aac", "-b:a", "192k", str(segment)])
seg = probe(segment)
W, H, FPS = seg["w"], seg["h"], seg["fps"]
N_FRAMES = int(round(seg["dur"] * FPS))
frame0 = next(read_frames(segment, W, H))
print(f"{W}x{H} ({'vertical' if H >= W else 'horizontal'}), {FPS:.2f} fps, {seg['dur']:.1f} s, {N_FRAMES} frames "
      f"({time.time() - t0:.0f} s)")

In [ ]:
#@title C2d · Prepare the background (removes you from the image if needed)
t0 = time.time()
bgp = Path(BACKGROUND_IMAGE)
assert bgp.exists(), f"Can't find {bgp}. Check BACKGROUND_IMAGE in Step 0."
bg = Image.open(bgp).convert("RGB")

# Fit the image to the video: fill the frame, crop the overflow from the centre.
ar_img, ar_vid = bg.width / bg.height, W / H
if abs(ar_img - ar_vid) / ar_vid > 0.03:
    print(f"Note: your image is {bg.width}x{bg.height} and the video is {W}x{H}. "
          "The image will be cropped to fit. For the best result, make it the same shape as the video.")
scale = max(W / bg.width, H / bg.height)
bg = bg.resize((math.ceil(bg.width * scale), math.ceil(bg.height * scale)), Image.LANCZOS)
left, top = (bg.width - W) // 2, (bg.height - H) // 2
bg = np.array(bg.crop((left, top, left + W, top + H)))

if BACKGROUND_HAS_ME:
    print("Removing you from the background image...")
    import cv2
    pm = person_mask(bg) > 40
    k = max(9, int(0.03 * min(W, H)) | 1)            # grow the hole a bit so no edge of you is left
    hole = cv2.dilate(pm.astype(np.uint8) * 255, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
    # LaMa works best around 1–1.3 MP, so fill at that size and paste only the hole back.
    f = min(1.0, 1280 / max(W, H))
    sw, sh = int(W * f) // 8 * 8, int(H * f) // 8 * 8
    from simple_lama_inpainting import SimpleLama
    lama = SimpleLama()
    filled = lama(Image.fromarray(bg).resize((sw, sh), Image.LANCZOS),
                  Image.fromarray(hole).resize((sw, sh), Image.NEAREST))
    filled = np.array(filled.crop((0, 0, sw, sh)).resize((W, H), Image.LANCZOS)).astype(np.float32)
    soft = cv2.GaussianBlur(hole, (0, 0), k / 6).astype(np.float32)[..., None] / 255
    bg = (bg * (1 - soft) + filled * soft).clip(0, 255).astype(np.uint8)
    del lama; torch.cuda.empty_cache()

BG_CLEAN = OUT / "background_clean.png"
Image.fromarray(bg).save(BG_CLEAN)
print(f"Background ready ({time.time() - t0:.0f} s). Saved to {BG_CLEAN}")
print("If the empty room doesn't look right, ask your image AI for 'the same room, empty' instead, "
      "untick BACKGROUND_HAS_ME, and run Step 0 and Step 3 again.")
show_images([BG_CLEAN], 240)

In [ ]:
#@title C2d2 · Cut you out of every frame with MatAnyone 2 (the slow part)
t0 = time.time()
from matanyone2 import MatAnyone2, InferenceCore
from matanyone2.utils.inference_utils import gen_dilate, gen_erosion

# 1) A mask of you on the first frame, so MatAnyone knows who to follow.
m0 = person_mask(frame0)
first_mask = ((m0 > 127) * 255).astype(np.uint8)
Image.fromarray(first_mask).save(WORK / "first_mask.png")
overlay = frame0.copy(); overlay[first_mask == 0] = (overlay[first_mask == 0] * 0.25).astype(np.uint8)
print("This is who will be cut out (everything dark is background):")
show_images([overlay], 200)
assert first_mask.mean() > 2, "No person found in the first frame. Try a START where you're clearly in view."
import gc; gc.collect()

# 2) Run MatAnyone 2 frame by frame (same steps as its own video function, but streamed,
#    so a 1-minute 1080p video fits in Colab's memory).
model = MatAnyone2.from_pretrained("PeiqingYang/MatAnyone2")
proc = InferenceCore(model, device=DEV)
N_WARMUP = 10
short = min(W, H)
if short > MATTING_SIZE:
    nh, nw = int(H / short * MATTING_SIZE), int(W / short * MATTING_SIZE)
else:
    nh, nw = H, W

mask = gen_erosion(gen_dilate(first_mask, 10, 10), 10, 10)
mask_t = torch.from_numpy(mask).float().to(DEV)
if (nh, nw) != (H, W):
    mask_t = F.interpolate(mask_t[None, None], size=(nh, nw), mode="nearest")[0, 0]

ALPHA = WORK / "alpha.mkv"
aw = writer(ALPHA, W, H, FPS, gray=True)
plate_sum = torch.zeros(3, H, W, device=DEV); plate_cnt = torch.zeros(1, H, W, device=DEV)

def frames_with_warmup():
    for _ in range(N_WARMUP):
        yield frame0
    yield from read_frames(segment, W, H)

done, t_loop = 0, time.time()
with torch.inference_mode(), torch.amp.autocast(device_type=DEV.type, enabled=DEV.type == "cuda"):
    for ti, fr in enumerate(frames_with_warmup()):
        full = torch.from_numpy(fr.copy()).to(DEV).permute(2, 0, 1).float()
        img = full if (nh, nw) == (H, W) else F.interpolate(full[None], size=(nh, nw), mode="area")[0]
        img = img / 255.0
        if ti == 0:
            proc.step(img, mask_t, objects=[1])
            prob = proc.step(img, first_frame_pred=True)
        elif ti <= N_WARMUP:
            prob = proc.step(img, first_frame_pred=True)
        else:
            prob = proc.step(img)
        if ti < N_WARMUP:
            continue
        a = proc.output_prob_to_mask(prob).float().clamp(0, 1)
        if a.shape[-2:] != (H, W):
            a = F.interpolate(a[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0]
        a = a[None]
        # Remember what your real room looks like where you're not standing (used to clean your edges).
        bgm = (a < 0.02).float()
        plate_sum += full / 255.0 * bgm; plate_cnt += bgm
        aw.stdin.write((a[0] * 255 + 0.5).byte().cpu().numpy().tobytes())
        done += 1
        if done % 150 == 0:
            rate = done / (time.time() - t_loop)
            print(f"  {done}/{N_FRAMES} frames  ({rate:.1f} frames/s, about {(N_FRAMES - done) / rate / 60:.1f} min left)")
aw.stdin.close(); aw.wait()

torch.save({"sum": plate_sum.cpu(), "cnt": plate_cnt.cpu()}, WORK / "plate.pt")
del model, proc; torch.cuda.empty_cache()
print(f"Cut-out done: {done} frames in {(time.time() - t0) / 60:.1f} min "
      f"({done / (time.time() - t_loop):.1f} frames/s).")

In [ ]:
#@title C2e · Put you in the new room (re-run after changing the realism sliders)
t0 = time.time()
with torch.inference_mode():
    # The new background, softly blurred like a real lens.
    Bn = torch.from_numpy(np.array(Image.open(BG_CLEAN).convert("RGB"))).to(DEV).permute(2, 0, 1).float() / 255
    Bn = gblur(Bn, BACKGROUND_BLUR * H / 1920 * 1.0)
    px = max(W, H) / 1920                                  # scale effects with the video size
    Bwrap = big_blur(Bn, 14 * px)

    # Your real room (from Step 4), filled in where you were always standing.
    pl = torch.load(WORK / "plate.pt")
    cnt = pl["cnt"].to(DEV); plate = pl["sum"].to(DEV) / cnt.clamp(min=1)
    valid = (cnt >= 3).float()
    fill_s = 40 * px
    filled = big_blur(plate * valid, fill_s) / big_blur(valid, fill_s).clamp(min=1e-3)
    plate = plate * valid + filled * (1 - valid)
    valid_soft = big_blur(valid, 3 * px)

    def to_ycc(x):
        y = 0.2126 * x[0] + 0.7152 * x[1] + 0.0722 * x[2]
        return torch.stack([y, (x[2] - y) / 1.8556, (x[0] - y) / 1.5748])
    def from_ycc(c):
        y, cb, cr = c
        r = y + 1.5748 * cr; b = y + 1.8556 * cb
        g = (y - 0.2126 * r - 0.0722 * b) / 0.7152
        return torch.stack([r, g, b])

    def foreground(I, a):
        """Your colours with the old room's colour removed from the edges."""
        solved = (I - (1 - a) * plate) / a.clamp(min=0.05)
        spread = gblur(I * a, 4 * px) / gblur(a, 4 * px).clamp(min=1e-4)
        w = ((a - 0.15) / 0.35).clamp(0, 1) * valid_soft
        Fg = w * solved + (1 - w) * spread
        Fg = torch.where(a > 0.97, I, Fg)
        return Fg.clamp(0, 1)

    # Colour match, worked out once (so it never flickers): nudge your average colour towards the room's.
    a0 = next(read_frames(ALPHA, W, H, gray=True))
    a0 = torch.from_numpy(a0.copy()).to(DEV).permute(2, 0, 1).float() / 255
    I0 = torch.from_numpy(frame0.copy()).to(DEV).permute(2, 0, 1).float() / 255
    fg_px = to_ycc(foreground(I0, a0))[:, a0[0] > 0.95]
    bg_ycc = to_ycc(Bn).flatten(1)
    if fg_px.shape[1] < 100:
        fg_px = to_ycc(I0).flatten(1)                      # fallback: whole first frame
    shift = (bg_ycc.mean(1) - fg_px.mean(1)) * COLOR_MATCH
    shift[0] *= 0.5                                        # brightness: only half, so your face stays readable
    shift = shift.view(3, 1, 1)

    # Grain: measure your camera's noise on the real room, add the same amount to the new one.
    hp = (I0 - gblur(I0, 1.0)).mean(0)
    bgpix = hp[(a0[0] < 0.02)]
    noise = float(1.4826 * bgpix.abs().median()) if bgpix.numel() > 1000 else 0.01
    noise = min(noise, 0.05) * GRAIN

    out_path = OUT / f"{src.stem}_new_background.mp4"
    vw = writer(out_path, W, H, FPS, audio_from=segment if seg["audio"] else None)
    n = 0
    for fr, af in zip(read_frames(segment, W, H), read_frames(ALPHA, W, H, gray=True)):
        I = torch.from_numpy(fr.copy()).to(DEV).permute(2, 0, 1).float() / 255
        a = torch.from_numpy(af.copy()).to(DEV).permute(2, 0, 1).float() / 255
        if EDGE_TIGHTEN > 0:
            a = ((a - EDGE_TIGHTEN) / (1 - EDGE_TIGHTEN)).clamp(0, 1)
        Fg = foreground(I, a)
        if COLOR_MATCH > 0:
            Fg = from_ycc(to_ycc(Fg) + shift).clamp(0, 1)
        bgf = Bn
        if noise > 0:
            bgf = (Bn + torch.randn(1, H, W, device=DEV) * noise).clamp(0, 1)
        comp = a * Fg + (1 - a) * bgf
        if LIGHT_WRAP > 0:
            wrap = (big_blur(1 - a, 10 * px) * a * LIGHT_WRAP).clamp(0, 1)
            comp = comp * (1 - wrap) + Bwrap * wrap
        vw.stdin.write((comp.clamp(0, 1) * 255 + 0.5).byte().permute(1, 2, 0).contiguous().cpu().numpy().tobytes())
        n += 1
        if n == 1:
            sample = (comp.clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
    vw.stdin.close(); vw.wait()

# Side-by-side check video (original | new), smaller so it's quick to open.
compare = OUT / f"{src.stem}_compare.mp4"
ch = min(H, 960) // 2 * 2
run(["ffmpeg", "-v", "error", "-y", "-i", str(segment), "-i", str(out_path), "-filter_complex",
     f"[0:v]scale=-2:{ch}[a];[1:v]scale=-2:{ch}[b];[a][b]hstack[v]", "-map", "[v]", "-map", "0:a?",
     *X264, "-c:a", "aac", "-shortest", str(compare)])

fin = probe(out_path)
print(f"Done in {time.time() - t0:.0f} s.")
print(f"Your video: {out_path}")
print(f"Check video (original | new): {compare}")
print(f"{fin['w']}x{fin['h']}, {fin['fps']:.2f} fps, {fin['dur']:.1f} s, sound: {'yes' if fin['audio'] else 'no'}")
show_images([frame0, sample], 220)

In [ ]:
#@title Part C3 · Soft grade → final edit.mp4 for Claude (≈2–4 min)
import os, shutil
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
swapped = bool(CHANGE_BACKGROUND and "out_path" in globals() and Path(str(out_path)).exists())
final_input = str(out_path) if swapped else SWAP_INPUT
if swapped:
    shutil.copy(compare, f"{RES}/compare-background.mp4")
flags = f'--slug "{SLUG}" --root "{ROOT}" --input "{final_input}"' + (" --swapped" if swapped else "") + (" --omni" if OMNI_FLASH else "")
!cd /content && python colab_finish.py finish {flags}
print(f'''
DONE → watch {RES}/preview-720p.mp4 on your phone, then Claude Code: "finish {SLUG}" (same folder link).''')